# Sequence Deduplication with CD-HIT

This notebook applies CD-HIT at 40% pairwise sequence identity to remove redundant sequences from the positive and negative sets independently. One representative per cluster is retained.

**Input:** `distinct_positive_sequences.csv`, `distinct_negative_sequences.csv`  
**Output:** `distinct_positive_clean.csv`, `distinct_negative_clean.csv`

In [ ]:
!sudo apt-get install -y cd-hit

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
pos_df = pd.read_csv('distinct_positive_sequences.csv')
neg_df = pd.read_csv('distinct_negative_sequences.csv')

print(f'Positive sequences: {len(pos_df)}')
print(f'Negative sequences: {len(neg_df)}')
print(f'\nColumns: {pos_df.columns.tolist()}')

In [ ]:
def csv_to_fasta(df, seq_col, label, output_path):
    with open(output_path, 'w') as f:
        for i, row in df.iterrows():
            f.write(f'>{label}_{i}\n{row[seq_col]}\n')
    print(f'Written {len(df)} sequences to {output_path}')

csv_to_fasta(pos_df, 'sequence', 'pos', 'train_pos.fasta')
csv_to_fasta(neg_df, 'sequence', 'neg', 'train_neg.fasta')

!cat train_pos.fasta train_neg.fasta > train_all.fasta
!echo 'Combined line count:' && wc -l train_all.fasta

In [ ]:
!cd-hit -i train_all.fasta -o train_clustered.fasta -c 0.4 -n 2 -M 8000 -T 4

In [ ]:
def parse_clstr(clstr_path):
    clusters = []
    current = []
    with open(clstr_path) as f:
        for line in f:
            if line.startswith('>Cluster'):
                if current:
                    clusters.append(current)
                current = []
            else:
                if '>' in line:
                    seq_id = line.split('>')[1].split('...')[0]
                    current.append(seq_id)
    if current:
        clusters.append(current)
    return clusters

clusters = parse_clstr('train_clustered.fasta.clstr')
all_ids = [sid for c in clusters for sid in c]
multi_member = [c for c in clusters if len(c) > 1]

print(f'Total sequences input:       {len(all_ids)}')
print(f'Total clusters formed:       {len(clusters)}')
print(f'Clusters with >1 member:     {len(multi_member)}')
print(f'Redundant sequences removed: {len(all_ids) - len(clusters)}')

print('\nExample redundant clusters:')
for c in multi_member[:5]:
    print(' ', c)

In [ ]:
representative_ids = set(c[0] for c in clusters)

pos_df_idx = pos_df.copy().reset_index(drop=True)
neg_df_idx = neg_df.copy().reset_index(drop=True)

pos_keep = [i for i in range(len(pos_df_idx)) if f'pos_{i}' in representative_ids]
neg_keep = [i for i in range(len(neg_df_idx)) if f'neg_{i}' in representative_ids]

pos_clean = pos_df_idx.iloc[pos_keep].reset_index(drop=True)
neg_clean = neg_df_idx.iloc[neg_keep].reset_index(drop=True)

print(f'Positive: {len(pos_df)} → {len(pos_clean)} ({(len(pos_df)-len(pos_clean))/len(pos_df)*100:.1f}% removed)')
print(f'Negative: {len(neg_df)} → {len(neg_clean)} ({(len(neg_df)-len(neg_clean))/len(neg_df)*100:.1f}% removed)')

pos_clean.to_csv('distinct_positive_clean.csv', index=False)
neg_clean.to_csv('distinct_negative_clean.csv', index=False)
print('\nClean CSVs saved')

In [ ]:
print(f'Positive (acetylated):     {len(pos_clean)}')
print(f'Negative (non-acetylated): {len(neg_clean)}')
print(f'Pos:Neg ratio:             {len(pos_clean)/len(neg_clean):.2f}:1')
print(f'Total training corpus:     {len(pos_clean)+len(neg_clean)}')

print('\nSequence lengths:')
print('  Positive:', pos_clean['sequence'].str.len().value_counts().sort_index().to_dict())
print('  Negative:', neg_clean['sequence'].str.len().value_counts().sort_index().to_dict())

print(f'\nDuplicates remaining — Pos: {pos_clean["sequence"].duplicated().sum()}, Neg: {neg_clean["sequence"].duplicated().sum()}')

In [ ]:
def aa_composition(sequences):
    all_aa = ''.join(sequences)
    total = len(all_aa)
    counts = Counter(all_aa)
    return {aa: round(counts[aa]/total*100, 2) for aa in sorted(counts)}

pos_comp = aa_composition(pos_clean['sequence'].tolist())
neg_comp = aa_composition(neg_clean['sequence'].tolist())

print(f'{"AA":<6} {"Positive":>10} {"Negative":>10} {"Difference":>12}')
print('-' * 42)
for aa in sorted(set(list(pos_comp) + list(neg_comp))):
    p = pos_comp.get(aa, 0)
    n = neg_comp.get(aa, 0)
    diff = p - n
    flag = ' <<<' if abs(diff) > 1.5 else ''
    print(f'{aa:<6} {p:>10} {n:>10} {diff:>+12.2f}{flag}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
categories = ['Positive sequences', 'Negative sequences']
before = [len(pos_df), len(neg_df)]
after  = [len(pos_clean), len(neg_clean)]

x = np.arange(len(categories))
w = 0.35
ax.bar(x - w/2, before, w, label='Before CD-HIT', color='steelblue', alpha=0.8)
ax.bar(x + w/2, after,  w, label='After CD-HIT',  color='coral',     alpha=0.8)

for i, (b, a) in enumerate(zip(before, after)):
    pct = (b - a) / b * 100
    ax.annotate(f'{pct:.0f}% removed', xy=(i + w/2, a),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Number of sequences')
ax.set_title('Sequence redundancy removed by CD-HIT (40% identity threshold)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('redundancy_figure.png', dpi=200)
plt.show()